# Linear Regression with PyTorch
Learns the relationship `y ≈ 2x - 1` from 6 data points.
Pure PyTorch equivalent of the Keras Sequential + Dense approach.

In [1]:
import numpy as np
import torch
import torch.nn as nn

print(f"PyTorch version: {torch.__version__}")
# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

PyTorch version: 2.2.2
Using device: cpu


In [2]:
# Ground truth: y ≈ 2x - 1

xs = np.array([-1, 0, 1, 2, 3, 4], dtype=np.float32)
ys = np.array([-3, -1, 1, 3, 5, 7], dtype=np.float32)

# Convert to PyTorch tensors and move to device
# unsqueeze(1) reshapes (6,) -> (6, 1) - PyTorch expects (batch, features)
X = torch.from_numpy(xs).unsqueeze(1).to(device)
Y = torch.from_numpy(ys).unsqueeze(1).to(device)

print(f"X shape: {X.shape}, Y shape: {Y.shape}")

X shape: torch.Size([6, 1]), Y shape: torch.Size([6, 1])


In [3]:
# nn.Linear(1, 1): one input feature -> one output (equivalent to Dense(1))
# Internally holds weight w and bias b -> computes y = xW^T + b
# nn.Sequential: mirrors Keras Sequential -> ordered layer stack

model = nn.Sequential(
    nn.Linear(in_features=1, out_features=1)  # single neuron, linear activation
).to(device)

# Print architecture and parameter count
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {total_params}")

Sequential(
  (0): Linear(in_features=1, out_features=1, bias=True)
)
Total trainable parameters: 2


In [4]:
# MSELoss: equivalent to keras.losses.MeanSquaredError()
# SGD with lr=0.1: same optimizer and learning rate as the Keras version

criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

In [5]:
# Each epoch: forward pass -> compute loss -> backprop -> update weights

epochs = 500
history = []

model.train()  # set model to training mode (enables dropout/batchnorm if present)

for epoch in range(1, epochs + 1):
    optimizer.zero_grad()       # clear gradients from previous step
    y_pred = model(X)           # forward pass
    loss = criterion(y_pred, Y) # compute MSE loss
    loss.backward()             # backpropagation: compute gradients
    optimizer.step()            # update weights via SGD
    history.append(loss.item())

print(f"Stopped at epoch {len(history)} | final MSE: {history[-1]:.6f}")

Stopped at epoch 500 | final MSE: 0.000000


In [6]:
# Inspect learned parameters - should converge to w≈2, b≈-1
# model[0] accesses the nn.Linear layer inside nn.Sequential

w = model[0].weight.item()
b = model[0].bias.item()
print(f"Learned weight (w): {w:.4f} | bias (b): {b:.4f}")
print("Expected: w ≈ 2.0000 | b ≈ -1.0000")

Learned weight (w): 2.0000 | bias (b): -1.0000
Expected: w ≈ 2.0000 | b ≈ -1.0000


In [7]:
# Inference: predict y for x=10 -> expected ≈ 19.0 (2*10 - 1)
# torch.no_grad(): disables gradient tracking - saves memory during inference

model.eval()  # set model to eval mode

x_new = torch.tensor([[10.0]], dtype=torch.float32).to(device)

with torch.no_grad():
    prediction = model(x_new)

print(f"Prediction for x=10: {prediction.item():.4f}  (expected ≈ 19.0)")

Prediction for x=10: 19.0000  (expected ≈ 19.0)


In [ ]:
# Evaluate on training data: compute MSE and MAE manually

model.eval()

with torch.no_grad():
    y_pred = model(X)
    mse = nn.MSELoss()(y_pred, Y).item()
    mae = nn.L1Loss()(y_pred, Y).item()   # L1Loss = Mean Absolute Error

print(f"Train MSE: {mse:.4f} | Train MAE: {mae:.4f}")

Train MSE: 0.0000 | Train MAE: 0.0000
